# MiniCS + vLLM on Google Colab — a programmatic knowledge base & dataset studio

This notebook is one of the MiniCS examples. It runs the whole
MiniCS library *programmatically* (no UI, no desktop app) on a free Colab GPU
runtime, backed by **vLLM** serving two OpenAI-compatible models in the
background:

| Server | Default model | Port | OpenAI surface |
| --- | --- | --- | --- |
| Chat LLM | `Qwen/Qwen2.5-1.5B-Instruct` | `:8001` | `/v1/chat/completions` |
| Embeddings | `BAAI/bge-m3` (`--task embed`, 1024 dims) | `:8002` | `/v1/embeddings` |

**What you will do, end to end:**

1. Install vLLM + `minichat-studio` (MiniCS as a pure library).
2. Launch both vLLM servers in the background (`run_vllm.sh`).
3. Configure a MiniCS home (`MINICS_HOME=/content/minics-home`) entirely from
   Python: `AppContext`, config patching, embedding-dimension probing.
4. Build the **documental database**: import markdown documents, index them
   into ChromaDB (vectors) and the Ladybug entity graph.
5. Run **hybrid retrieval** (vector + graph fused with RRF).
6. Chat with the vLLM model **grounded** in your documents, with `[n]` citations.
7. **Author datasets**: generate a grounded ChatML entry, suggest tags, evaluate
   quality, then curate a collection and **export to JSONL**.
8. Clean up (stop the servers).

Everything is local to the Colab VM: documents, SQLite, ChromaDB, the graph
and the model weights. Nothing is sent anywhere except the localhost calls to
your own vLLM servers.

> **Runtime:** `Runtime → Change runtime type → T4 GPU`. A T4 (16 GB) fits the
> defaults comfortably. First run downloads ~4 GB of model weights.

## 1 · Runtime check

Confirm the GPU is visible. If `nvidia-smi` fails, switch the runtime type
before continuing.

In [ ]:
!nvidia-smi
import sys, platform
print("python:", sys.version)
print("platform:", platform.platform())

## 2 · Install vLLM, PyTorch and MiniCS — from a terminal

> **Do this from a terminal, not from notebook cells** (Colab: *File → New
> terminal*, or SSH). The dependency install must land in the VM's system
> Python before any server starts. If you prefer cells, prefix each line
> with `!` — but the terminal is the reliable path.

```bash
# 1. uv — fast parallel resolver (vLLM's officially recommended installer)
pip install -U uv

# 2. Remove the Colab-preinstalled torch family (its CUDA builds do not
#    match what vLLM needs — the source of the classic
#    "PyTorch and TorchAudio were compiled with different CUDA versions" crash)
uv pip uninstall --system torch torchvision torchaudio

# 3. Install a pinned, mutually consistent cu130 stack
#    (CUDA 13 needs driver >= 580 — current Colab GPU images have it;
#     on a CUDA-12 image use .../whl/cu128 and an older vLLM release instead)
uv pip install --system     torch==2.11.0     torchvision==0.26.0     torchaudio==2.11.0     --index-url https://download.pytorch.org/whl/cu130

# 4. vLLM + MiniCS (as published PyPI packages — no source installs)
uv pip install --system vllm "minichat-studio>=0.3.2"
```

The next cell **verifies** the result: the whole torch stack must import
cleanly (same CUDA build), and the VM's driver must support the CUDA variant
you installed. Everything after this point assumes these four commands
succeeded.

In [ ]:
# Verify the dependency install from the terminal before launching anything.
import torch, torchaudio, torchvision, vllm
print("torch:       ", torch.__version__)
print("torchaudio:  ", torchaudio.__version__)
print("torchvision: ", torchvision.__version__)
print("vllm:        ", vllm.__version__)

import importlib.metadata as im
for pkg in ("minichat-studio", "chromadb"):
    print(f"{pkg:20s} {im.version(pkg)}")

# The driver on the Colab VM must support the CUDA build just installed
# (CUDA 13 wheels need driver >= 580). Fail here, with a clear message,
# instead of deep inside `vllm serve` later.
import re, subprocess
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
driver_cuda = re.search(r"CUDA Version:\s*(\d+\.\d+)", smi)
if driver_cuda:
    print("driver CUDA: ", driver_cuda.group(1))
    if int(torch.version.cuda.split(".")[0]) > int(driver_cuda.group(1).split(".")[0]):
        raise RuntimeError(
            f"torch was built for CUDA {torch.version.cuda} but the VM's driver "
            f"only supports {driver_cuda.group(1)}. Reinstall the stack for the "
            "driver's CUDA variant (see the install section above: use a cu128 "
            "index and an older vLLM release on CUDA-12 images)."
        )
else:
    print("warning: could not read the driver CUDA version from nvidia-smi")

## 3 · Launch the two vLLM servers in the background

One vLLM process serves one model, so the launcher starts **two**:

- `vllm serve Qwen/Qwen2.5-1.5B-Instruct --port 8001` → the chat LLM
  (16K context window),
- `vllm serve BAAI/bge-m3 --port 8002` → the embedding model (vLLM
  auto-detects it as a pooling model from its architecture and serves
  OpenAI-style `/v1/embeddings`).

The next cell writes **`run_vllm.sh`** — the launcher script shipped in the
repository next to this notebook
(`examples/colab-vllm/run_vllm.sh`, also fetchable straight from GitHub if
you prefer: `!wget -q https://raw.githubusercontent.com/jasonjimnz/minics/main/examples/colab-vllm/run_vllm.sh`).
It stops any previous servers first, starts both with `nohup`, polls
`/health` until each server is live (first start downloads the weights —
allow several minutes), and prints where the logs are. `bash run_vllm.sh
stop` tears everything down (the notebook's final cell).

In [ ]:
%%writefile run_vllm.sh
#!/usr/bin/env bash
# ---------------------------------------------------------------------------
# run_vllm.sh — start vLLM in the background with TWO servers:
#
#   1. a chat LLM   (default: Qwen/Qwen2.5-1.5B-Instruct)   on :8001
#   2. an embedding model (default: BAAI/bge-m3)             on :8002
#
# Dependency install (uv, torch stack, vLLM) is documented in the notebook's
# markdown and must be run from a terminal BEFORE this script — it is not
# handled here.
#
#   bash run_vllm.sh          # start both servers, wait until healthy
#   bash run_vllm.sh stop     # kill both servers
#
# Logs: $VLLM_LOG_DIR (default /tmp/minics-vllm-logs)
# ---------------------------------------------------------------------------
set -euo pipefail

LLM_MODEL="${LLM_MODEL:-Qwen/Qwen2.5-1.5B-Instruct}"
EMBED_MODEL="${EMBED_MODEL:-BAAI/bge-m3}"
LLM_PORT="${LLM_PORT:-8001}"
EMBED_PORT="${EMBED_PORT:-8002}"
LLM_GPU_FRAC="${LLM_GPU_FRAC:-0.45}"     # T4-friendly split; tune to your GPU
EMBED_GPU_FRAC="${EMBED_GPU_FRAC:-0.30}"
MAX_MODEL_LEN="${MAX_MODEL_LEN:-16384}"  # 16K context: room for grounding + long replies
LOG_DIR="${VLLM_LOG_DIR:-/tmp/minics-vllm-logs}"
HEALTH_TIMEOUT="${HEALTH_TIMEOUT:-900}"  # first start downloads the weights

mkdir -p "$LOG_DIR"

log()  { echo "[vllm-launcher] $*"; }
fail() { echo "[vllm-launcher] ERROR: $*" >&2; exit 1; }

stop_all() {
  log "stopping any running vLLM servers..."
  pkill -f "vllm serve" 2>/dev/null || true
  pkill -f "from vllm"   2>/dev/null || true
  sleep 3
}

wait_healthy() {
  local port="$1" name="$2" waited=0
  until curl -sf "http://127.0.0.1:${port}/health" > /dev/null 2>&1; do
    sleep 5
    waited=$((waited + 5))
    if ! pgrep -f "vllm serve" > /dev/null; then
      echo "---- last log lines (${name}) ----" >&2
      tail -n 40 "${LOG_DIR}/${name}.log" >&2 || true
      fail "${name} process died — see ${LOG_DIR}/${name}.log"
    fi
    if [ "$waited" -ge "$HEALTH_TIMEOUT" ]; then
      fail "${name} not healthy after ${HEALTH_TIMEOUT}s — see ${LOG_DIR}/${name}.log"
    fi
  done
  log "${name} is healthy on :${port} (waited ${waited}s)"
}

if [ "${1:-}" = "stop" ]; then
  stop_all
  log "stopped."
  exit 0
fi

command -v curl > /dev/null || fail "curl is required"
command -v vllm > /dev/null || fail "vLLM is not installed — run the dependency install commands from the notebook's markdown section (terminal)"

# Preflight: torch / torchaudio / torchvision must be built for the same
# CUDA version, or `vllm serve` dies on import.
PY="$(command -v python3 || command -v python)"
if ! "$PY" -c "import torch, torchaudio, torchvision" > /dev/null 2>&1; then
  fail "torch/torchaudio/torchvision CUDA mismatch (or missing). Fix with:
    uv pip uninstall --system torch torchvision torchaudio
    uv pip install --system torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu130
    uv pip install --system vllm
  (see the notebook's install section), then re-run this script."
fi
log "preflight OK: the torch stack imports cleanly"

stop_all

log "starting chat LLM:  ${LLM_MODEL}  on :${LLM_PORT} (gpu ${LLM_GPU_FRAC})"
nohup vllm serve "${LLM_MODEL}" \
  --port "${LLM_PORT}" \
  --gpu-memory-utilization "${LLM_GPU_FRAC}" \
  --max-model-len "${MAX_MODEL_LEN}" \
  > "${LOG_DIR}/llm.log" 2>&1 &

# Note: no --task/--runner flag needed - vLLM auto-detects bge-m3 as a
# pooling (embedding) model from its architecture.
log "starting embedder:  ${EMBED_MODEL}  on :${EMBED_PORT} (gpu ${EMBED_GPU_FRAC})"
nohup vllm serve "${EMBED_MODEL}" \
  --port "${EMBED_PORT}" \
  --gpu-memory-utilization "${EMBED_GPU_FRAC}" \
  --max-model-len 8192 \
  > "${LOG_DIR}/embed.log" 2>&1 &

wait_healthy "${LLM_PORT}"   "llm"
wait_healthy "${EMBED_PORT}" "embed"

log "both servers are up:"
log "  chat LLM    -> http://127.0.0.1:${LLM_PORT}/v1   (${LLM_MODEL})"
log "  embeddings  -> http://127.0.0.1:${EMBED_PORT}/v1 (${EMBED_MODEL})"
log "logs in ${LOG_DIR}"


In [ ]:
!chmod +x run_vllm.sh
!bash run_vllm.sh

Both endpoints are now live. You can watch the logs any time:

```bash
!tail -n 5 /tmp/minics-vllm-logs/llm.log /tmp/minics-vllm-logs/embed.log
```

## 4 · Smoke-test the OpenAI endpoints

MiniCS is OpenAI-protocol first, so before wiring it up we verify that both
servers answer like OpenAI would: list models, one chat completion, one
embedding (note the returned vector size — MiniCS will probe it again
automatically in the next section).

In [ ]:
from openai import OpenAI

llm = OpenAI(base_url="http://127.0.0.1:8001/v1", api_key="EMPTY")
emb = OpenAI(base_url="http://127.0.0.1:8002/v1", api_key="EMPTY")

print("chat models:   ", [m.id for m in llm.models.list().data])
print("embed models:  ", [m.id for m in emb.models.list().data])

completion = llm.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "Say hello in one short sentence."}],
    max_tokens=48,
)
print("chat reply:    ", completion.choices[0].message.content.strip())

vector = emb.embeddings.create(model="BAAI/bge-m3", input=["dimension probe"])
print("embedding dim: ", len(vector.data[0].embedding))

## 5 · Configure MiniCS programmatically

MiniCS is a library first — the desktop app and CLI are thin layers over a
single application context, `minics.services.context.AppContext`. Here we:

1. point `MINICS_HOME` at `/content/minics-home` (the whole studio — SQLite,
   ChromaDB, graph, documents — lives inside that folder),
2. create the context and patch its config straight to the vLLM endpoints,
3. **probe the embedding dimension** from the live endpoint (Chroma
   collections are created with an explicit dimensionality, so it must be
   exact — bge-m3 is 1024),
4. mark setup complete and check readiness.

This replaces the interactive `minics setup` wizard — same config keys, same
`~/.minics`-style home, just written from code.

In [ ]:
import os

os.environ["MINICS_HOME"] = "/content/minics-home"

from minics.services.context import get_context
from minics.llm.client import probe_embedding_dimension

ctx = get_context("/content/minics-home")
ctx.ensure_ready()

# Deep-merged into config.json and persisted atomically.
ctx.update_config({
    "llm": {
        "base_url": "http://127.0.0.1:8001/v1",
        "api_key": "EMPTY",
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "temperature": 0.3,
        "max_tokens": 8192,  # 16K context window: room for grounding + long replies
    },
    "embedding": {
        "base_url": "http://127.0.0.1:8002/v1",
        "api_key": "EMPTY",
        "model": "BAAI/bge-m3",
        "batch_size": 32,
    },
})

# The vector store needs the exact dimensionality — probe the live endpoint.
dimension = probe_embedding_dimension(config=ctx.config)
ctx.update_config({"embedding": {"dimension": dimension}})
ctx.config_manager.mark_setup_complete()

print("embedding dimension:", dimension)
print("ready:", ctx.config.is_ready(), "| missing:", ctx.config.missing_requirements())
print("home:", ctx.paths.root)

## 6 · Build the documental database

Documents are MiniCS's source of truth for grounding. We write two small
markdown documents to disk (in real life these would be PDF/DOCX/TXT/LaTeX —
`import_file` converts them to clean markdown automatically), then:

1. **import** each file (`ctx.documents.import_file`) — stored as original +
   cleaned markdown in `MINICS_HOME/documents/`,
2. **index** it (`ctx.indexer.index_document`) — chunked along headings,
   embedded in batches, written to ChromaDB + the `document_chunks` table,
3. **wire the graph** (`ctx.graph_indexer.index_document`) — heuristic entity
   extraction builds mentions and co-occurrence edges in the Ladybug store.

No LLM is required for any of this — only the embedding endpoint.

In [ ]:
from pathlib import Path

docs_dir = Path("/content/sample_docs")
docs_dir.mkdir(parents=True, exist_ok=True)

DOC_A = """\
# Hybrid Retrieval: Vectors + Graphs + RRF

## Why single-signal retrieval is not enough

Dense vector search embeds a query and a corpus of chunks into the same
vector space and returns the nearest neighbours by cosine similarity. It is
excellent at paraphrase, but it has two classic failure modes: lexical
precision loss on rare identifiers, and topology blindness - a vector knows
nothing about how concepts relate across the document.

## The graph side

A knowledge graph built from the same corpus stores entities and the chunks
where they appear, plus co-occurrence edges between entities that show up
together. Query terms are matched against entity names; the entities' chunks
are ranked by how strongly they mention them, with neighbour expansion
walking one or two hops to pull in related concepts the query never mentioned.

## Reciprocal Rank Fusion

RRF is rank-based: each retriever produces a ranked list and every chunk
accumulates 1 / (k + rank) for each list it appears in (k is commonly 60).
Because RRF uses ranks rather than raw scores, it never needs to compare
incompatible score scales. A chunk both retrievers agree on tends to surface
near the top, and if one side degrades the other still answers.
"""

DOC_B = """\
# MiniCS Field Manual

## The knowledge base

Documents (PDF, DOCX, TXT, Markdown, LaTeX) are imported and converted to
clean markdown, then split into chunks along headings. Each chunk is embedded
into a ChromaDB vector collection while entity extraction feeds a graph
database recording which entities appear in which chunks.

## Entries and datasets

A dataset is a container for entries. An entry is a ChatML conversation with
system, user and assistant messages, carrying tags, a lifecycle status
(draft, review, approved) and an automatic version history.

## Grounded authoring

The authoring toolbox can enhance fields, suggest tags, and evaluate entries
against six quality dimensions: faithfulness, relevance, completeness,
clarity, groundedness and safety. It can also generate a brand-new entry
about a topic, grounded in retrieved document chunks.

## Export

Collections curate approved entries and export to ChatML, JSONL, Alpaca,
ShareGPT or Markdown - byte-identical from the UI, the CLI or Python.
"""

(docs_dir / "hybrid-retrieval-primer.md").write_text(DOC_A, encoding="utf-8")
(docs_dir / "minics-field-manual.md").write_text(DOC_B, encoding="utf-8")

for path in sorted(docs_dir.glob("*.md")):
    document = ctx.documents.import_file(path, source="colab")
    vector_info = ctx.indexer.index_document(document.id)
    graph_info = ctx.graph_indexer.index_document(document.id)
    print(f"{document.title}: {vector_info['chunks']} chunks, "
          f"{graph_info['entities']} entity mentions")

print("\nlibrary status:", ctx.documents.stats())
print("graph stats:   ", ctx.graph.stats())

## 7 · Hybrid retrieval — vector + graph fused with RRF

`ctx.retriever.retrieve` runs **both** signals — Chroma cosine search and
Ladybug graph topology — and fuses the rankings with Reciprocal Rank Fusion.
Each side degrades gracefully if the other is unavailable. The result exposes
per-chunk scores, the sources that selected it, and a ready-made
`context_text()` grounding block numbered `[1]…[n]` for prompts.

In [ ]:
result = ctx.retriever.retrieve("How does Reciprocal Rank Fusion work?", top_k=4)

print("diagnostics:", result.diagnostics)
print("used rag:", result.used_rag, "| used graph:", result.used_graph)
print("-" * 72)
for position, chunk in enumerate(result.chunks, start=1):
    print(f"[{position}] {chunk.document_title} - {chunk.heading or 'chunk'}")
    print(f"    fused={chunk.fused_score:.4f}  vector={chunk.vector_score:.3f}  "
          f"graph={chunk.graph_score:.3f}  sources={chunk.sources}")
    print("    entities:", ", ".join(chunk.entities[:6]) or "-")
print("-" * 72)
print(result.context_text(max_chars=900)[:600], "\n...")

## 8 · Grounded chat with the vLLM model

`ctx.chat` persists conversations with three independent switches:

- **`use_rag`** — retrieve semantically similar chunks,
- **`use_graph`** — retrieve chunks selected through the entity graph,
- **`use_grounding`** — actually inject what was retrieved into the prompt.

Retrieval and injection are separate on purpose: you can *preview* what would
be used without forcing it into the answer. With grounding on, the system
prompt receives the `[n]`-numbered context block and the model is instructed
to cite sources inline.

In [ ]:
conversation = ctx.chat.create(
    "Colab demo",
    system_prompt="Answer concisely for a technical audience.",
    use_rag=True,
    use_graph=True,
    use_grounding=True,
)

reply = ctx.chat.send(conversation.id, "Why is hybrid retrieval more robust than vector search alone?")

print(reply["content"])
print("-" * 72)
print("citations:")
for citation in reply["citations"]:
    print(f"  [{citation['ref']}] {citation['title']} (score {citation['score']:.4f})")
print("usage:", reply["usage"])

## 9 · Dataset authoring — generate, tag, evaluate

The authoring toolbox turns the knowledge base into a dataset factory:

- `generate_entry` drafts a complete ChatML entry about a topic, grounded in
  retrieved chunks from your documents,
- `suggest_tags` proposes tags for an entry (optionally merging them in),
- `evaluate_entry` scores the entry across six quality dimensions and stores
  the evaluation on the entry.

Every persisted edit bumps the entry **version** automatically.

In [ ]:
dataset = ctx.datasets.create(
    "Colab Grounded QA",
    description="Entries generated on Colab, grounded in the sample documents",
    tags=["colab", "demo"],
)

# 1. Generate a grounded entry about a topic.
entry = ctx.authoring.generate_entry(
    topic="How does MiniCS ground LLM answers in documents?",
    dataset=dataset.id,
    system_hint="Be precise and mention the hybrid retrieval pipeline.",
    use_grounding=True,
    tags=["rag"],
)
print("generated entry:", entry.public_id)
print("  user:      ", entry.user[:140], "...")
print("  assistant: ", entry.assistant[:220], "...")

# 2. Suggest tags (and merge them into the entry).
tag_suggestion = ctx.authoring.suggest_tags(entry.id, max_tags=5, add=True)
print("suggested tags:", tag_suggestion["tags"])

# 3. Evaluate quality (persisted on the entry).
evaluation = ctx.authoring.evaluate_entry(entry.id, use_grounding=True)
print(f"quality score: {evaluation.score:.2f} / 1.0")
print("  strengths:  ", evaluation.strengths)
print("  issues:     ", evaluation.issues)

## 10 · Curate a collection and export

Collections curate entries for export. The exporter is shared by the UI, CLI
and Python, so the output is **byte-identical** everywhere. We export as
JSONL (one ChatML conversation per line) and save it into the Colab VM —
swap the last line for `files.download(path)` to pull it into your browser.

In [ ]:
from google.colab import files  # on a local Jupyter, just skip the download call

collection = ctx.collections.create(
    "Colab export",
    description="Approved demo entries",
    dataset=dataset.id,
)
collection = ctx.collections.add_entries(collection.id, [entry.id])

payload = ctx.collections.export(collection.id, fmt="jsonl", include_metadata=True)
out_path = "/content/minics-export.jsonl"
Path(out_path).write_text(payload, encoding="utf-8")

print(f"exported {len(payload.splitlines())} entries -> {out_path} ({len(payload)} bytes)")
print(payload.splitlines()[0][:400], "...")
# files.download(out_path)   # uncomment to download the JSONL to your machine

## 11 · Cleanup

Stop both vLLM servers and close MiniCS's stores (Chroma + Ladybug + SQLite)
cleanly.

In [ ]:
!bash run_vllm.sh stop
ctx.close()
print("demo finished — MiniCS home kept at /content/minics-home")

## Troubleshooting

| Symptom | Fix |
| --- | --- |
| `RuntimeError: Detected that PyTorch and TorchAudio were compiled with different CUDA versions` when running `run_vllm.sh` | The terminal install from §2 was skipped or partially run. Re-run its four commands (uninstall torch family → pinned cu130 reinstall → vLLM), check the verification cell passes, then re-run the launcher — its preflight catches the mismatch before starting the servers. |
| `CUDA out of memory` during startup | Lower the GPU fractions (`LLM_GPU_FRAC=0.35 EMBED_GPU_FRAC=0.25 bash run_vllm.sh`) or use smaller models (e.g. `Qwen/Qwen2.5-0.5B-Instruct`, `BAAI/bge-small-en-v1.5` — dimension 384). |
| Launcher says the process died | Read the tail printed from `/tmp/minics-vllm-logs/llm.log` or `embed.log`; on Colab the usual culprit is the GPU not being enabled. |
| `Embedding dimension mismatch` warnings | The stored dimension no longer matches the endpoint (you changed the embedding model). Wipe `/content/minics-home/vectordb` and re-run the config + indexing cells. |
| Embedding endpoint returns 404 on `/v1/embeddings` | The served model is not a pooling model, or the request hit the chat port (8001). Embeddings live on :8002 — check the launcher line for `bge-m3` (vLLM auto-detects pooling models; on older vLLM you may need `--task embed`, on the newest ones `--runner pooling`). |
| `kb`-style search returns few results | Only two tiny documents are indexed here — add more documents (Section 6) before judging retrieval quality. |
| Chat replies are cut off | Defaults are already generous (`--max-model-len 16384` in the launcher, `max_tokens: 8192` in the MiniCS config). If you still hit limits, raise `MAX_MODEL_LEN` before starting the servers or `ctx.update_config({"llm": {"max_tokens": 16384}})`. |

## Next steps

- **Serve this knowledge base to a coding agent**: see
  `examples/mcp-knowledge-base` — the same home can be exposed as
  an MCP server for OpenCode.
- **Use the desktop app on the same home**: install MiniCS locally, set
  `MINICS_HOME` to the same folder (or download it from Colab), and open the
  UI — datasets, entries, documents and indexes are all there.
- **Scale the corpus**: `import_file` handles PDF, DOCX, TXT and LaTeX;
  `ctx.indexer.reindex_all()` rebuilds everything after a config change.